In [6]:
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.linear_model import LinearRegression
import yfinance as yf

In [7]:
end = dt.date.today()
start = dt.date.today() - dt.timedelta(days=365*5)
ticker = ['HDFCBANK.NS','^NSEI','^INDIAVIX','^NSEBANK']
df = yf.download(tickers=ticker,start=start,end=end,auto_adjust=True)['Close']

# Dropping Nans
df[df.isna().any(axis=1)]
df.dropna(inplace=True)

# Features Engineering
df['hdfc_returns'] = np.log(df['HDFCBANK.NS']/df['HDFCBANK.NS'].shift(1))
df['hdfc_lagged'] = df['hdfc_returns'].shift(1)
df['nse_returns'] = np.log(df['^NSEI']/df['^NSEI'].shift(1))
df['bank_returns'] = np.log(df['^NSEBANK']/df['^NSEBANK'].shift(1))
df['volatility'] = df['hdfc_returns'].rolling(20).std()
df['momentum'] = df['HDFCBANK.NS'].pct_change(5)

# Cleaning
df.drop(columns=['^NSEBANK','^NSEI','HDFCBANK.NS'],inplace=True)
df.dropna(inplace=True)
df.columns = df.columns.str.lower()
df.rename(columns = {'^indiavix':'vix'},inplace=True)
columns = ['hdfc_returns','hdfc_lagged','nse_returns','bank_returns','volatility','momentum','vix']
df = df[[col for col in columns if col in df.columns]]


[*********************100%***********************]  4 of 4 completed


In [8]:
# ADF test

from statsmodels.tsa.stattools import adfuller
def adf_test(series, name=""):
    result = adfuller(series.dropna())

    print(f"\nADF Test → {name}")
    print(f"ADF Statistic: {result[0]:.2f}")
    print(f"p-value: {result[1]:.2f}")

    if result[1] < 0.05:
        print("→ Stationary ✅")
    else:
        print("→ Non-stationary ❌")

for col in df.columns:
    adf_test(df[col], col)


ADF Test → hdfc_returns
ADF Statistic: -33.81
p-value: 0.00
→ Stationary ✅

ADF Test → hdfc_lagged
ADF Statistic: -33.73
p-value: 0.00
→ Stationary ✅

ADF Test → nse_returns
ADF Statistic: -35.19
p-value: 0.00
→ Stationary ✅

ADF Test → bank_returns
ADF Statistic: -16.22
p-value: 0.00
→ Stationary ✅

ADF Test → volatility
ADF Statistic: -3.42
p-value: 0.01
→ Stationary ✅

ADF Test → momentum
ADF Statistic: -6.93
p-value: 0.00
→ Stationary ✅

ADF Test → vix
ADF Statistic: -3.81
p-value: 0.00
→ Stationary ✅


In [9]:
# Splitting Data
train = int(len(df) * 0.8)
train_df = df.iloc[:train]
test_df = df.iloc[train:]

# Define X and y
x_train = train_df.drop(columns=['vix'])
y_train = train_df['hdfc_returns']

# Train model using SKlearn
model = LinearRegression()
model = model.fit(x_train, y_train)

In [12]:
import statsmodels.api as sm

x_train_sm = sm.add_constant(x_train)
model_sm = sm.OLS(y_train, x_train_sm).fit()

model_sm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           hdfc_returns   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.279e+29
Date:                Fri, 27 Mar 2026   Prob (F-statistic):               0.00
Time:                        14:52:00   Log-Likelihood:                 32544.
No. Observations:                 961   AIC:                        -6.507e+04
Df Residuals:                     954   BIC:                        -6.504e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const         4.727e-16   4.69e-17     10.088      0.000    3.81e-16    5.65e-16
hdfc_returns     1.0000   1.89e-15   5.28e+14      0.000       1.000       1.000
hdfc_lagged   2.762e-16   1.32e-15      0.209      0.834   -2.32e-15    2.87e-15
nse_returns  -2.358e-16   3.46e-15     -0.068      0.946   -7.02e-15    6.54e-15
bank_returns -4.573e-16   3.18e-15     -0.144      0.886   -6.69e-15    5.78e-15
volatility    1.096e-16   3.45e-15      0.032      0.975   -6.67e-15    6.89e-15
momentum      2.433e-16   6.58e-16      0.370      0.712   -1.05e-15    1.53e-15
==============================================================================
Omnibus:                      108.661   Durbin-Watson:                   0.000
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              869.511
Skew:                          -0.101   Prob(JB):                    1.54e-189
Kurtosis:                       7.656   Cond. No.                         285.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [14]:
import pandas as pd

summary_df = pd.DataFrame({
    "Feature": model_sm.params.index,
    "Coefficient": model_sm.params.values,
    "p-value": model_sm.pvalues.values
})

print(summary_df)

        Feature   Coefficient       p-value
0         const  4.726932e-16  8.293259e-23
1  hdfc_returns  1.000000e+00  0.000000e+00
2   hdfc_lagged  2.762448e-16  8.344389e-01
3   nse_returns -2.358259e-16  9.455992e-01
4  bank_returns -4.573432e-16  8.855405e-01
5    volatility  1.095876e-16  9.746963e-01
6      momentum  2.433172e-16  7.116279e-01


In [15]:
print("R²:", model_sm.rsquared)
print("Adj R²:", model_sm.rsquared_adj)
print("Prob(F):", model_sm.f_pvalue)

print("\nCoefficients:")
print(model_sm.params)

print("\np-values:")
print(model_sm.pvalues)

R²: 1.0
Adj R²: 1.0
Prob(F): 0.0

Coefficients:
const           4.726932e-16
hdfc_returns    1.000000e+00
hdfc_lagged     2.762448e-16
nse_returns    -2.358259e-16
bank_returns   -4.573432e-16
volatility      1.095876e-16
momentum        2.433172e-16
dtype: float64

p-values:
const           8.293259e-23
hdfc_returns    0.000000e+00
hdfc_lagged     8.344389e-01
nse_returns     9.455992e-01
bank_returns    8.855405e-01
volatility      9.746963e-01
momentum        7.116279e-01
dtype: float64
